In [ ]:
from kiwipiepy import Kiwi
import pandas as pd
from pathlib import Path
from pilos.preprocessing.comments import run_comment_preprocessing

# 경로 상수
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "data"
DATA_NAME = "comments_cleaned.json"

# 사용자 사전
USER_DICTIONARY = {
    "외인": "NNG",
    "저점": "NNG",
    "고점": "NNG",
}

# 판다스 디스플레이 설정
pd.set_option("display.max_colwidth", None)

# 형태소분석기 객체 소환
kiwi = Kiwi()

# 사용자 사전 등록
for word, tag in USER_DICTIONARY.items():
    kiwi.add_user_word(word, tag)

# 전처리 완료된 df 가져오기
processed_df = run_comment_preprocessing(
    input_path=DATA_DIR/"raw"/"until_2026-07-23_삼성전자.json",
    output_path=DATA_DIR/"processed"/"comment.json",
)

In [ ]:
# 토크나이저 실행
def tokenize_comment(text):
    '''
    텍스트를 토큰화하여 반환합니다
    '''
    tokens = kiwi.tokenize(text)
    return [
        {
            "form":token.form,
            "tag":token.tag
        }
        for token in tokens
    ]
def format_tokens_for_review(tokens):
    """
    토큰 목록을 CSV 검수용 문자열로 변환합니다.
    """
    return " | ".join(
        f"{token['form']}/{token['tag']}"
        for token in tokens
    )

In [ ]:
# 샘플 데이터추출
sample_df = processed_df.sample(n=100,random_state=5)
# 샘플데이터 토큰화
sample_df["kiwi_tokens"] = (
    sample_df["comment_text_raw"]
    .apply(tokenize_comment)
)
# 샘플데이터 토큰 포맷팅(리뷰용)
sample_df["kiwi_result"] = (
    sample_df["kiwi_tokens"]
    .apply(format_tokens_for_review)
)
